# 4. Session Comparison — does anticipatory licking narrow with learning, and does DCZ perturb it?

Reads the batch outputs from `batch_lickproc.py`:

- `all_sessions_lickmetrics.csv` — one summary row per recording
- `all_sessions_trajectory.csv` — rolling within-session rows (window 30, step 8)
- `<rec>/<rec>_lickmetrics.h5` — the per-session lick-rate-vs-position curves

**Cohort:** JSY083, JSY084 · 8 baseline days (B1–B8) → Saline → DCZ100 → DCZ200
(one session per condition, no replicates → descriptive + effect sizes + within-session
trial-level spread, not group stats).

**Theory:** across baseline days the anticipatory licking should concentrate in the
last stretch before the reward zone — spatially (`lick_spread_cm` ↓, `lick_median_cm` →
RZ, `frac_approach_licks_in_rz` ↑, `anticipatory_index` ↑) and temporally
(`first_consummatory_lick_latency_s` ↓, `psth_anticipatory_lead_s` sharpens).
Saline should look like trained baseline; DCZ (dose-dependent) should broaden it back out.

In [ ]:
import os, sys, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm as mplcm
import h5py

import datetime as _dt

MODULE_DIR = r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation\09.LickingBehavior"
sys.path.insert(0, MODULE_DIR)
from lick_metrics import trial_cluster_bootstrap, bootstrap_pvalue

# ---------------- CONFIG ----------------
DATASET_ROOT = os.environ.get("LICK_DATASET_ROOT",
                              r"F:\dlc\lick-detector_v0-JSY-2026-06-16\videos")
# every run writes into a fresh dated folder -- new figures are never mixed
# in with an old run's, nothing to hunt through to tell which are current
RUN_DATE = _dt.date.today().isoformat()
FIGDIR = os.path.join(DATASET_ROOT, f"session_comparison_figures_{RUN_DATE}")
os.makedirs(FIGDIR, exist_ok=True)

SESSIONS   = [f"B{i}" for i in range(1, 9)] + ["Saline", "DCZ100", "DCZ200"]
BASELINE   = [f"B{i}" for i in range(1, 9)]
TRAINED    = ["B7", "B8"]                       # "trained baseline" reference
CONDITIONS = ["Saline", "DCZ100", "DCZ200"]
DOSE       = {"Saline": 0, "DCZ100": 100, "DCZ200": 200}
ANIMAL_COL = {"JSY083": "tab:blue", "JSY084": "tab:red"}

# metrics that should move as licking narrows (arrow = direction with learning)
KEY_METRICS = {
    "anticipatory_index":                "↑  (RZ vs neutral lick-rate contrast; approach licks ONLY -- spatial, position-occupancy-normalised)",
    "psth_anticipatory_index":            "↑  (pre-valve black-screen vs far-baseline lick rate; temporal counterpart -- CAN see consummatory_pre_reward)",
    "frac_approach_licks_in_rz":          "↑  (fraction of approach licks in last 30 cm)",
    "lick_median_cm":                     "↑ → RZ  (median approach-lick position)",
    "lick_spread_cm":                     "↓  (SD of approach-lick position)",
    "approaching_reward_per_trial":       "↑  (anticipatory licks / trial, while still running)",
    "consummatory_pre_reward_per_trial":  "↑  (anticipatory licks / trial, in the black screen -- temporal counterpart to the row above)",
    "rz_lick_rate_hz":                    "~ / ↑  (occupancy-normalised rate in RZ)",
    "first_consummatory_lick_latency_s":  "↓  (black-screen → first consummatory lick)",
    "psth_anticipatory_lead_s":           "↓ sharpens  (pre-valve ramp lead time)",
    "trial_duration_s_median":            "↓  (teleport → reward time; engagement/efficiency, independent of licking)",
}

# locomotor + consummatory readouts -- independent of licking, so they can
# tell apart "DCZ changes anticipation specifically" from "DCZ changes
# general movement" or "DCZ changes consumption/motivation"
CONTROL_METRICS = {
    "run_speed_vr_cms":                   "~  (general locomotion -- confound check)",
    "decel_onset_cm":                     "↑ → RZ  (motor anticipation: later/closer deceleration onset)",
}
# session-level only (bout duration/rate isn't a natural rolling-window
# quantity, so no within-session SD from T -- compared with a simpler plot)
BOUT_METRICS = {
    "drink_bout_duration_s_median":       "~  (consummatory: how long it drinks)",
    "drink_bout_lick_rate_hz_median":     "~  (consummatory: how fast it licks once drinking)",
}
plt.rcParams["figure.dpi"] = 110

def save(fig, name, retries=4):
    # F:\ has intermittent transient write hiccups (OSError Errno 22) under
    # nbclient -- unrelated to figure content; a short retry clears it.
    import time
    for attempt in range(retries):
        try:
            fig.savefig(os.path.join(FIGDIR, name), dpi=120, bbox_inches="tight")
            return
        except OSError:
            if attempt == retries - 1:
                raise
            time.sleep(0.5)

## Load batch outputs

In [ ]:
M = pd.read_csv(os.path.join(DATASET_ROOT, "all_sessions_lickmetrics.csv"))
T = pd.read_csv(os.path.join(DATASET_ROOT, "all_sessions_trajectory.csv"))

M["session_label"] = pd.Categorical(M["session_label"], categories=SESSIONS, ordered=True)
M = M.sort_values(["animal", "session_label"]).reset_index(drop=True)
T["session_label"] = pd.Categorical(T["session_label"], categories=SESSIONS, ordered=True)

ANIMALS = sorted(M.animal.unique())
print(f"{len(M)} sessions  |  animals: {ANIMALS}")
missing = [(a, s) for a in ANIMALS for s in SESSIONS
           if not ((M.animal == a) & (M.session_label == s)).any()]
print("missing sessions:", missing or "none")

# QC overview
qc = M[["animal", "session_label", "n_trials", "n_led_ok_trials", "led_flagged_trials",
        "led_blocked_runs", "sync_resid_sd_ms", "n_licks"]].copy()
qc["sync_resid_sd_ms"] = qc["sync_resid_sd_ms"].round(1)
qc

## 1. Baseline learning across days

Does the licking narrow onto the reward zone over B1–B8? Saline / DCZ sessions
are plotted on the same axis (shaded) for reference.

In [ ]:
xpos = {s: i for i, s in enumerate(SESSIONS)}
mets = list(KEY_METRICS)
fig, axs = plt.subplots(6, 2, figsize=(13, 19), sharex=True)
for ax in axs.ravel()[len(mets):]:
    ax.axis("off")
for ax, k in zip(axs.ravel(), mets):
    for a in ANIMALS:
        g = M[M.animal == a].sort_values("session_label")
        x = [xpos[s] for s in g.session_label]
        ax.plot(x, g[k], "-o", ms=4, color=ANIMAL_COL.get(a, "tab:gray"), label=a)
    # baseline trend line (both animals pooled)
    gb = M[M.session_label.isin(BASELINE)]
    if gb[k].notna().sum() > 2:
        bx = gb.session_label.map(xpos).to_numpy(float)
        p = np.polyfit(bx[np.isfinite(gb[k])], gb[k].to_numpy()[np.isfinite(gb[k])], 1)
        xs = np.array([0, 7]); ax.plot(xs, np.polyval(p, xs), ":", color="k", lw=1)
    ax.axvspan(xpos["Saline"] - 0.5, xpos["DCZ200"] + 0.5, color="tab:orange", alpha=0.08)
    ax.set_title(f"{k}\n{KEY_METRICS[k]}", fontsize=8.5)
axs[0, 0].legend(fontsize=8)
for ax in axs[-1]:
    ax.set_xticks(range(len(SESSIONS)))
    ax.set_xticklabels(SESSIONS, rotation=45, ha="right", fontsize=8)
fig.suptitle("Session-level licking metrics — baseline learning + DCZ", y=1.005)
plt.tight_layout(); save(fig, "1_baseline_learning_scalars.png"); plt.show()

### Approach vs. consummatory balance across baseline learning

Does anticipatory licking shift from **approach** licking (while still
running, position-locked, in `KEY_METRICS` above) to **consummatory
pre-reward** licking (during the black screen, waiting for the valve — a
purely time-locked signal that the approach-based metrics structurally can't
see) as the animal learns the corridor? Baseline days only (B1–B8); left =
raw licks/trial for each; right = the fraction of all anticipatory licks
(approach + consummatory_pre_reward) that happen in the black screen — one
number tracking the spatial→temporal shift, independent of how much the
animal licks overall.

In [ ]:
fig, axs = plt.subplots(2, 2 * len(ANIMALS), figsize=(6 * len(ANIMALS), 8), squeeze=False)
for j, a in enumerate(ANIMALS):
    g = M[(M.animal == a) & (M.session_label.isin(BASELINE))].sort_values("session_label")
    bx = np.arange(len(g))
    approach_total = g.approaching_reward_per_trial + g.approaching_nonreward_per_trial
    consum_pre = g.consummatory_pre_reward_per_trial
    frac = consum_pre / (consum_pre + approach_total)

    ax = axs[0, 2 * j]
    ax.plot(bx, approach_total, "-o", ms=5, color="tab:orange", label="approach (total)")
    ax.plot(bx, consum_pre, "-o", ms=5, color="tab:cyan", label="consummatory_pre_reward")
    ax.set_xticks(bx); ax.set_xticklabels(g.session_label, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("licks / trial"); ax.set_title(f"{a} — approach vs consummatory"); ax.legend(fontsize=7)

    ax2 = axs[0, 2 * j + 1]
    ax2.plot(bx, frac, "-o", ms=5, color="tab:purple")
    if np.isfinite(frac).sum() > 2:
        p = np.polyfit(bx[np.isfinite(frac)], frac.to_numpy()[np.isfinite(frac)], 1)
        ax2.plot(bx, np.polyval(p, bx), ":", color="k", lw=1)
    ax2.set_ylim(0, 1)
    ax2.set_xticks(bx); ax2.set_xticklabels(g.session_label, rotation=45, ha="right", fontsize=8)
    ax2.set_ylabel("fraction in black screen"); ax2.set_title(f"{a} — consummatory share of anticipation")

    # composition as %-of-trial-licks stacked bar, baseline only -- same 4
    # categories as the full-dataset composition figure below, but restricted
    # to B1-B8 and normalised to 100% so the SHAPE of the shift is visible
    # regardless of how much the animal licks overall that day
    cats = ["approaching_nonreward_per_trial", "approaching_reward_per_trial",
           "consummatory_pre_reward_per_trial", "consummatory_post_reward_per_trial"]
    ccol = ["tab:gray", "tab:orange", "tab:cyan", "tab:blue"]
    tot = g[cats].sum(axis=1)
    bot = np.zeros(len(g))
    for c, col in zip(cats, ccol):
        pct = 100 * g[c] / tot
        axs[1, 2 * j].bar(bx, pct, 0.8, bottom=bot, color=col, label=c.replace("_per_trial", ""))
        bot += pct.to_numpy()
    axs[1, 2 * j].set_xticks(bx); axs[1, 2 * j].set_xticklabels(g.session_label, rotation=45, ha="right", fontsize=8)
    axs[1, 2 * j].set_ylabel("% of licks"); axs[1, 2 * j].set_title(f"{a} — composition (% of trial licks)")
    axs[1, 2 * j].legend(fontsize=6.5)
    axs[1, 2 * j + 1].axis("off")
plt.tight_layout(); save(fig, "1_approach_vs_consummatory_baseline.png"); plt.show()

### First anticipatory-lick position across baseline days

Not just the session median (`first_lick_cm_median`, already in the scalar
trajectory above) — the trial-by-trial scatter itself, one panel per baseline
day in order, sharing one y-scale per animal. A rising/tightening scatter
across days is more convincing evidence of "the animal's first response gets
more precise/confident" than a moving median alone, since a stable median can
hide a distribution that's still scattered.

In [ ]:
def load_first_licks(rec):
    p = glob.glob(os.path.join(DATASET_ROOT, "*", rec, f"{rec}_lickmetrics.h5"))
    if not p:
        return None
    with h5py.File(p[0], "r") as f:
        return pd.DataFrame({k: f["first_licks"][k][:] for k in f["first_licks"]})

fig, axs = plt.subplots(len(ANIMALS), len(BASELINE),
                        figsize=(2.6 * len(BASELINE), 2.6 * len(ANIMALS)), squeeze=False)
for i, a in enumerate(ANIMALS):
    g = M[(M.animal == a) & (M.session_label.isin(BASELINE))].set_index("session_label")
    rz = g.reward_zone_cm.median()
    fl_cache = {s: load_first_licks(g.loc[s, "recording"]) for s in BASELINE if s in g.index}
    ymax = max((np.nanmax(fl.first_lick_cm.to_numpy()) for fl in fl_cache.values() if fl is not None),
              default=1.0)
    for j, s in enumerate(BASELINE):
        ax = axs[i, j]
        fl = fl_cache.get(s)
        if fl is not None:
            ax.scatter(fl.trial, fl.first_lick_cm, s=8, alpha=0.5, color=ANIMAL_COL.get(a, "gray"))
            ax.axhline(rz, color="tab:red", ls="--", lw=0.8)
        ax.set_ylim(0, ymax * 1.05)
        if i == 0:
            ax.set_title(s, fontsize=9)
        if j == 0:
            ax.set_ylabel(f"{a}\nfirst-lick (cm)", fontsize=8)
        else:
            ax.set_yticklabels([])
        if i == len(ANIMALS) - 1:
            ax.set_xlabel("trial", fontsize=7)
fig.suptitle("First approach-lick position, trial-by-trial, B1 -> B8", y=1.02)
plt.tight_layout(); save(fig, "1_first_lick_position_baseline.png"); plt.show()

### Lick-rate-vs-position and occupancy — how the curve changes with experience

11 overlapping lines per animal (B1 → DCZ200) made it hard to see the trend by
eye. Two views instead, both ordered B1 (top) → DCZ200 (bottom):

- **Heatmap** — each row is one session, colour = the value at that position.
  Narrowing / shifting shows up directly as the coloured band moving or
  tightening down the rows.
- **Small multiples** — the same per-session curves as individual panels in
  session order, sharing one y-scale per animal, for a closer look at any one
  session without the others' lines in the way.

`dwell_s` is a session TOTAL, divided by that session's trial count so
sessions with more trials (or the freeze-heavy later ones) are comparable.

In [ ]:
def load_curve(rec):
    p = glob.glob(os.path.join(DATASET_ROOT, "*", rec, f"{rec}_lickmetrics.h5"))
    if not p:
        return None
    with h5py.File(p[0], "r") as f:
        return pd.DataFrame({k: f["curve"][k][:] for k in f["curve"]})

def load_speed(rec):
    p = glob.glob(os.path.join(DATASET_ROOT, "*", rec, f"{rec}_lickmetrics.h5"))
    if not p:
        return None
    with h5py.File(p[0], "r") as f:
        return pd.DataFrame({k: f["speed_accel"][k][:] for k in f["speed_accel"]})

def session_matrix(g, curves_by_session, value_fn, binw=5.0):
    # (sessions, bin_cm, matrix[n_sessions, n_bins]) on ONE common position
    # grid. Each session's own bins can differ very slightly (corridor cap =
    # reward_zone_cm + 30, and reward_zone_cm varies ~0.5 cm across sessions),
    # so every curve is linearly interpolated onto a shared grid before
    # stacking into an image.
    have = {s: c for s, c in curves_by_session.items() if c is not None}
    if not have:
        return [], np.array([]), np.zeros((0, 0))
    hi = max(c.bin_cm.max() for c in have.values())
    bin_cm = np.arange(binw / 2, hi + binw / 2, binw)
    sessions = [s for s in g.session_label if s in have]
    Hm = np.full((len(sessions), len(bin_cm)), np.nan)
    for i, s in enumerate(sessions):
        c = have[s]
        Hm[i] = np.interp(bin_cm, c.bin_cm, value_fn(c), left=np.nan, right=np.nan)
    return sessions, bin_cm, Hm


def heatmap_panel(ax, sessions, bin_cm, Hm, rz, title, cbar_label, cmap="viridis", fig=None):
    if len(sessions) == 0:
        ax.axis("off"); return
    vmax = np.nanpercentile(Hm, 95) if np.isfinite(Hm).any() else 1.0
    im = ax.imshow(Hm, aspect="auto", cmap=cmap, vmin=0, vmax=vmax,
                   extent=[bin_cm[0], bin_cm[-1], len(sessions) - 0.5, -0.5])
    ax.axvline(rz, color="w", ls="--", lw=1.2)
    ax.set_yticks(range(len(sessions))); ax.set_yticklabels(sessions, fontsize=8)
    ax.set_xlabel("corridor position (cm)"); ax.set_title(title, fontsize=9.5)
    if fig is not None:
        fig.colorbar(im, ax=ax, label=cbar_label, shrink=0.8)


def small_multiples_panel(axs_col, g, curves_by_session, value_fn, rz, ylabel):
    ymax = 0.0
    for s, c in curves_by_session.items():
        if c is not None:
            ymax = max(ymax, float(np.nanmax(value_fn(c))))
    for i, s in enumerate(SESSIONS):
        ax = axs_col[i]
        c = curves_by_session.get(s)
        if c is not None:
            col = {"Saline": "tab:green", "DCZ100": "tab:orange", "DCZ200": "tab:red"}.get(s, "tab:blue")
            y = pd.Series(value_fn(c)).rolling(3, center=True, min_periods=1).mean()
            ax.plot(c.bin_cm, y, color=col, lw=1.3)
            ax.axvline(rz, color="tab:red", ls="--", lw=0.7, alpha=0.6)
        ax.set_ylim(0, ymax * 1.05 if ymax > 0 else 1)
        ax.set_ylabel(s, fontsize=7, rotation=0, ha="right", va="center", labelpad=2)
        ax.set_yticks([])

In [ ]:
# per-animal caches, built once and reused by both the heatmap and the
# small-multiples cells below -- dwell is normalised to per-trial up front
# (session_matrix / small_multiples_panel just see whatever's in `dwell_s`)
curves_cache, dwell_cache, speeds_cache = {}, {}, {}
for a in ANIMALS:
    g = M[M.animal == a].sort_values("session_label")
    ntr = {row.session_label: max(row.n_trials, 1) for _, row in g.iterrows()}
    curves = {row.session_label: load_curve(row.recording) for _, row in g.iterrows()}
    curves_cache[a] = curves
    dwell_cache[a] = {s: (c.assign(dwell_s=c.dwell_s / ntr[s]) if c is not None else None)
                     for s, c in curves.items()}
    speeds_cache[a] = {row.session_label: load_speed(row.recording) for _, row in g.iterrows()}

# --- heatmaps: lick rate, occupancy/trial, speed -- one figure, 3 rows ---
fig, axs = plt.subplots(3, len(ANIMALS), figsize=(7 * len(ANIMALS), 13), squeeze=False)
for j, a in enumerate(ANIMALS):
    g = M[M.animal == a].sort_values("session_label")
    rz = g.reward_zone_cm.median()

    s1, b1, H1 = session_matrix(g, curves_cache[a], lambda c: c.rate_hz.to_numpy())
    heatmap_panel(axs[0, j], s1, b1, H1, rz, f"{a} — approach lick rate (Hz)", "Hz", fig=fig)

    s2, b2, H2 = session_matrix(g, dwell_cache[a], lambda c: c.dwell_s.to_numpy())
    heatmap_panel(axs[1, j], s2, b2, H2, rz, f"{a} — occupancy (dwell / trial, s)", "s", cmap="magma", fig=fig)

    s3, b3, H3 = session_matrix(g, speeds_cache[a], lambda c: c.speed_cm_s.to_numpy())
    heatmap_panel(axs[2, j], s3, b3, H3, rz, f"{a} — running speed (cm/s)", "cm/s", cmap="cividis", fig=fig)
plt.tight_layout(); save(fig, "1_lickrate_occupancy_speed_heatmap.png"); plt.show()

In [ ]:
# --- small multiples: lick rate and occupancy, one panel per session ---
for value_name, cache, fn, fname in [
    ("lick rate (Hz)", curves_cache, lambda c: c.rate_hz.to_numpy(), "1_lickrate_small_multiples.png"),
    ("occupancy (dwell/trial, s)", dwell_cache, lambda c: c.dwell_s.to_numpy(), "1_occupancy_small_multiples.png"),
]:
    fig, axs = plt.subplots(len(SESSIONS), len(ANIMALS),
                            figsize=(4.5 * len(ANIMALS), 1.35 * len(SESSIONS)),
                            squeeze=False, sharex=True)
    for j, a in enumerate(ANIMALS):
        g = M[M.animal == a].sort_values("session_label")
        rz = g.reward_zone_cm.median()
        small_multiples_panel(axs[:, j], g, cache[a], fn, rz, value_name)
        axs[0, j].set_title(a, fontsize=10)
    for ax in axs[-1]:
        ax.set_xlabel("corridor position (cm)")
    fig.suptitle(f"{value_name} — one panel per session, B1 (top) -> DCZ200 (bottom)", y=1.0)
    plt.tight_layout(); save(fig, fname); plt.show()

### Position-log freeze fraction, for reference

In [ ]:
fig, axs = plt.subplots(1, len(ANIMALS), figsize=(6.5 * len(ANIMALS), 3.6), squeeze=False)
for ax, a in zip(axs[0], ANIMALS):
    g = M[M.animal == a].sort_values("session_label")
    colors = mplcm.viridis(np.linspace(0, 1, len(g)))
    x = np.arange(len(g))
    ax.bar(x, g.pct_position_frozen, color=colors)
    ax.set_xticks(x); ax.set_xticklabels(g.session_label, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("% session frozen"); ax.set_title(f"{a} — position-log freeze fraction")
plt.tight_layout(); save(fig, "1_freeze_fraction_by_session.png"); plt.show()

In [ ]:
# B1 vs B8 (first vs last baseline day) + lick-category composition across sessions
fig, axs = plt.subplots(2, len(ANIMALS), figsize=(7 * len(ANIMALS), 9), squeeze=False)
cats = ["approaching_nonreward_per_trial", "approaching_reward_per_trial",
        "consummatory_pre_reward_per_trial", "consummatory_post_reward_per_trial"]
ccol = ["tab:gray", "tab:orange", "tab:cyan", "tab:blue"]
for j, a in enumerate(ANIMALS):
    g = M[M.animal == a].set_index("session_label")
    rz = g.reward_zone_cm.median()
    for lbl, cc in [("B1", "tab:gray"), ("B8", "tab:green")]:
        if lbl in g.index:
            c = load_curve(g.loc[lbl, "recording"])
            if c is not None:
                axs[0, j].plot(c.bin_cm, c.rate_hz.rolling(3, center=True, min_periods=1).mean(),
                               "-o", ms=3, color=cc, label=lbl)
    axs[0, j].axvline(rz, color="tab:red", ls="--", lw=1)
    axs[0, j].axvspan(rz - 30, rz, color="tab:red", alpha=0.06)
    axs[0, j].set_title(f"{a} — B1 vs B8"); axs[0, j].legend(fontsize=8)
    axs[0, j].set_xlabel("position (cm)"); axs[0, j].set_ylabel("lick rate (Hz)")

    gg = M[M.animal == a].sort_values("session_label")
    x = np.arange(len(gg)); bot = np.zeros(len(gg))
    for c, col in zip(cats, ccol):
        axs[1, j].bar(x, gg[c], 0.8, bottom=bot, color=col, label=c.replace("_per_trial", ""))
        bot += gg[c].to_numpy()
    axs[1, j].set_xticks(x); axs[1, j].set_xticklabels(gg.session_label, rotation=45, ha="right", fontsize=8)
    axs[1, j].set_ylabel("licks / trial"); axs[1, j].set_title(f"{a} — lick composition")
    axs[1, j].legend(fontsize=7)
plt.tight_layout(); save(fig, "1_b1_vs_b8_and_composition.png"); plt.show()

## 2. Within-session dynamics

The rolling trajectory (window 30 trials, step 8). Within baseline days this shows
whether narrowing also happens *within* a session; in the DCZ sessions it shows
whether the effect builds over the session as the drug takes hold.

In [ ]:
roll_mets = ["anticipatory_index", "frac_approach_licks_in_rz",
             "lick_spread_cm", "first_consummatory_lick_latency_s"]
fig, axs = plt.subplots(len(roll_mets), len(ANIMALS), figsize=(7 * len(ANIMALS), 12),
                        squeeze=False, sharex=True)
for i, k in enumerate(roll_mets):
    for j, a in enumerate(ANIMALS):
        ax = axs[i, j]
        g = T[T.animal == a]
        for s in SESSIONS:
            gs = g[g.session_label == s].sort_values("trial_mid")
            if not len(gs):
                continue
            is_cond = s in CONDITIONS
            col = {"Saline": "tab:green", "DCZ100": "tab:orange", "DCZ200": "tab:red"}.get(
                s, mplcm.Blues(0.3 + 0.6 * BASELINE.index(s) / 7) if s in BASELINE else "gray")
            ax.plot(gs.trial_mid, gs[k], "-o", ms=3, lw=1.8 if is_cond else 1,
                    color=col, alpha=0.9 if is_cond else 0.55,
                    label=s if (is_cond or s in ("B1", "B8")) else None)
        if i == 0:
            ax.set_title(a); ax.legend(fontsize=7, ncol=2)
        if j == 0:
            ax.set_ylabel(k, fontsize=9)
for ax in axs[-1]:
    ax.set_xlabel("trial (window mid)")
fig.suptitle("Within-session rolling trajectory  (blue = baseline B1→B8, green/orange/red = Saline/DCZ100/DCZ200)",
             y=1.002, fontsize=10)
plt.tight_layout(); save(fig, "2_rolling_trajectory.png"); plt.show()

In [ ]:
# within-session slope of anticipatory_index vs trial  (learning rate within a day)
def within_slope(df, k="anticipatory_index"):
    d = df.dropna(subset=[k, "trial_mid"])
    if len(d) < 3:
        return np.nan
    return np.polyfit(d.trial_mid, d[k], 1)[0]

rows = []
for (a, s), g in T.groupby(["animal", "session_label"], observed=True):
    rows.append(dict(animal=a, session_label=s,
                     slope_antic_index=within_slope(g, "anticipatory_index"),
                     slope_frac_rz=within_slope(g, "frac_approach_licks_in_rz"),
                     slope_spread=within_slope(g, "lick_spread_cm")))
S = pd.DataFrame(rows)
S["session_label"] = pd.Categorical(S.session_label, categories=SESSIONS, ordered=True)
S = S.sort_values(["animal", "session_label"])

fig, axs = plt.subplots(1, len(ANIMALS), figsize=(7 * len(ANIMALS), 4), squeeze=False, sharey=True)
for ax, a in zip(axs[0], ANIMALS):
    g = S[S.animal == a]
    x = np.arange(len(g))
    colv = ["tab:green" if s == "Saline" else "tab:orange" if s == "DCZ100"
            else "tab:red" if s == "DCZ200" else "tab:blue" for s in g.session_label]
    ax.bar(x, g.slope_antic_index, color=colv)
    bm = g[g.session_label.isin(BASELINE)].slope_antic_index
    ax.axhline(bm.mean(), color="k", ls=":", lw=1, label=f"baseline mean {bm.mean():+.4f}")
    ax.axhline(0, color="gray", lw=0.6)
    ax.set_xticks(x); ax.set_xticklabels(g.session_label, rotation=45, ha="right", fontsize=8)
    ax.set_title(f"{a} — within-session slope (anticipatory_index / trial)"); ax.legend(fontsize=8)
plt.tight_layout(); save(fig, "2_within_session_slope.png"); plt.show()
S

## 3. Chemogenetic silencing — DCZ vs Saline vs trained baseline

Trained-baseline reference = mean of B7–B8; error band = SD across B1–B8.
Within-session spread for each condition = SD across its rolling windows.

In [ ]:
def cond_table(metric):
    rows = []
    for a in ANIMALS:
        gm = M[M.animal == a].set_index("session_label")
        gt = T[T.animal == a]
        base = gm.loc[gm.index.isin(BASELINE), metric]
        trained = gm.loc[gm.index.isin(TRAINED), metric].mean()
        r = dict(animal=a, baseline_mean=base.mean(), baseline_sd=base.std(),
                 trained=trained)
        for s in CONDITIONS:
            r[s] = gm.loc[s, metric] if s in gm.index else np.nan
            wr = gt[gt.session_label == s][metric]
            r[f"{s}_within_sd"] = wr.std()
            r[f"{s}_vs_trained_z"] = ((r[s] - trained) / base.std()
                                      if base.std() and np.isfinite(r[s]) else np.nan)
        rows.append(r)
    return pd.DataFrame(rows)

fig, axs = plt.subplots(6, 2, figsize=(13, 19))
for ax in axs.ravel()[len(KEY_METRICS):]:
    ax.axis("off")
for ax, k in zip(axs.ravel(), KEY_METRICS):
    ct = cond_table(k)
    xlab = ["trained\nB7-8"] + CONDITIONS
    for _, r in ct.iterrows():
        y = [r.trained] + [r[s] for s in CONDITIONS]
        e = [r.baseline_sd] + [r[f"{s}_within_sd"] for s in CONDITIONS]
        ax.errorbar(range(4), y, yerr=e, marker="o", ms=5, capsize=3,
                    color=ANIMAL_COL.get(r.animal, "gray"), label=r.animal)
    ax.set_xticks(range(4)); ax.set_xticklabels(xlab, fontsize=8)
    ax.set_title(f"{k}\n{KEY_METRICS[k]}", fontsize=8.5)
axs[0, 0].legend(fontsize=8)
fig.suptitle("Condition comparison — trained baseline vs Saline / DCZ100 / DCZ200", y=1.005)
plt.tight_layout(); save(fig, "3_condition_comparison.png"); plt.show()

In [ ]:
# dose-response: metric vs DCZ dose, with trained-baseline band
fig, axs = plt.subplots(6, 2, figsize=(13, 19))
for ax in axs.ravel()[len(KEY_METRICS):]:
    ax.axis("off")
for ax, k in zip(axs.ravel(), KEY_METRICS):
    ct = cond_table(k).set_index("animal")
    for a in ANIMALS:
        if a not in ct.index:
            continue
        r = ct.loc[a]
        xs = [DOSE[s] for s in CONDITIONS]
        ys = [r[s] for s in CONDITIONS]
        ax.plot(xs, ys, "-o", ms=5, color=ANIMAL_COL.get(a, "gray"), label=a)
        ax.axhspan(r.trained - r.baseline_sd, r.trained + r.baseline_sd,
                   color=ANIMAL_COL.get(a, "gray"), alpha=0.10)
        ax.axhline(r.trained, color=ANIMAL_COL.get(a, "gray"), ls=":", lw=1)
    ax.set_xticks([0, 100, 200]); ax.set_xticklabels(["Saline", "DCZ100", "DCZ200"], fontsize=8)
    ax.set_title(k, fontsize=9)
axs[0, 0].legend(fontsize=8)
fig.suptitle("Dose–response  (dotted line + band = trained baseline mean ± SD)", y=1.005)
plt.tight_layout(); save(fig, "3_dose_response.png"); plt.show()

In [ ]:
# does the DCZ effect emerge over the session?  rolling anticipatory_index,
# conditions overlaid, baseline envelope for reference
fig, axs = plt.subplots(1, len(ANIMALS), figsize=(7 * len(ANIMALS), 4.5), squeeze=False, sharey=True)
for ax, a in zip(axs[0], ANIMALS):
    gt = T[T.animal == a]
    bl = gt[gt.session_label.isin(BASELINE)]
    if len(bl):
        q = bl.groupby("trial_mid")["anticipatory_index"].agg(["mean", "std"])
        ax.fill_between(q.index, q["mean"] - q["std"], q["mean"] + q["std"],
                        color="tab:blue", alpha=0.12, label="baseline ± SD")
        ax.plot(q.index, q["mean"], color="tab:blue", lw=1)
    for s, col in [("Saline", "tab:green"), ("DCZ100", "tab:orange"), ("DCZ200", "tab:red")]:
        gs = gt[gt.session_label == s].sort_values("trial_mid")
        if len(gs):
            ax.plot(gs.trial_mid, gs.anticipatory_index, "-o", ms=4, color=col, label=s)
    ax.set_xlabel("trial (window mid)"); ax.set_title(f"{a} — anticipatory_index over session")
    ax.legend(fontsize=8)
axs[0, 0].set_ylabel("anticipatory_index")
plt.tight_layout(); save(fig, "3_dcz_within_session_timecourse.png"); plt.show()

## 3b. Locomotor + consummatory controls

Is a DCZ effect specific to *anticipatory licking*, or does it show up in
general movement / reward consumption too? `run_speed_vr_cms` (position-
derived, ground truth) and `dist_decel_onset_to_reward_cm` (motor analogue of
anticipatory onset — independent of licking) separate a locomotor effect from
a licking-specific one; `drink_bout_duration_s_median` /
`drink_bout_lick_rate_hz_median` ask whether DCZ also blunts *consumption*
once the reward is earned, a different circuit from anticipation.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2), squeeze=False)
for ax, k in zip(axs[0], CONTROL_METRICS):
    ct = cond_table(k)
    xlab = ["trained\nB7-8"] + CONDITIONS
    for _, r in ct.iterrows():
        y = [r.trained] + [r[s] for s in CONDITIONS]
        e = [r.baseline_sd] + [r[f"{s}_within_sd"] for s in CONDITIONS]
        ax.errorbar(range(4), y, yerr=e, marker="o", ms=5, capsize=3,
                    color=ANIMAL_COL.get(r.animal, "gray"), label=r.animal)
    ax.set_xticks(range(4)); ax.set_xticklabels(xlab, fontsize=8)
    ax.set_title(f"{k}\n{CONTROL_METRICS[k]}", fontsize=8.5)
axs[0, 0].legend(fontsize=8)
fig.suptitle("Locomotor controls — trained baseline vs Saline / DCZ100 / DCZ200", y=1.03)
plt.tight_layout(); save(fig, "3b_control_metrics.png"); plt.show()

# bout metrics are session-level only (no natural rolling-window value) --
# trained baseline mean +- across-day SD as a band, condition points overlaid
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2), squeeze=False)
for ax, k in zip(axs[0], BOUT_METRICS):
    for a in ANIMALS:
        gm = M[M.animal == a].set_index("session_label")
        base = gm.loc[gm.index.isin(BASELINE), k]
        trained = gm.loc[gm.index.isin(TRAINED), k].mean()
        y = [trained] + [gm.loc[s, k] if s in gm.index else np.nan for s in CONDITIONS]
        ax.plot(range(4), y, "-o", ms=5, color=ANIMAL_COL.get(a, "gray"), label=a)
        ax.axhspan(trained - base.std(), trained + base.std(), color=ANIMAL_COL.get(a, "gray"), alpha=0.08)
    ax.set_xticks(range(4)); ax.set_xticklabels(["trained\nB7-8"] + CONDITIONS, fontsize=8)
    ax.set_title(f"{k}\n{BOUT_METRICS[k]}", fontsize=8.5)
axs[0, 0].legend(fontsize=8)
fig.suptitle("Consummatory (drinking-bout) controls — session-level, band = trained ± baseline SD", y=1.03)
plt.tight_layout(); save(fig, "3b_bout_metrics.png"); plt.show()

### TM-log vs VR-derived speed — alignment QC across all 22 sessions

Two independent measurements of running speed should agree if both logs are
trustworthy. `speed_tm_vr_r2` is the per-session fit quality (near 1 = good
agreement); `speed_tm_vr_lag_s` is the small time-shift found between the two
logs' clocks (should be a similar small constant across sessions, not
wildly different — a jump would mean the shared-wall-clock assumption in
`lick_io.read_tmlog` broke that day).

In [ ]:
qc2 = M[["animal", "session_label", "speed_tm_vr_r2", "speed_tm_vr_lag_s",
        "run_speed_vr_cms", "run_speed_tm_cms"]].copy()
print(f"R² across 22 sessions: median {qc2.speed_tm_vr_r2.median():.2f}  "
      f"range {qc2.speed_tm_vr_r2.min():.2f}-{qc2.speed_tm_vr_r2.max():.2f}")
print(f"lag across 22 sessions: median {qc2.speed_tm_vr_lag_s.median():.2f} s  "
      f"range {qc2.speed_tm_vr_lag_s.min():.2f}-{qc2.speed_tm_vr_lag_s.max():.2f} s")
low = qc2[qc2.speed_tm_vr_r2 < 0.5]
if len(low):
    print(f"low-agreement sessions (R²<0.5): {list(zip(low.animal, low.session_label))}")

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for a in ANIMALS:
    g = qc2[qc2.animal == a]
    axs[0].scatter(g.session_label.astype(str), g.speed_tm_vr_r2, color=ANIMAL_COL.get(a, "gray"), label=a)
    axs[1].scatter(g.session_label.astype(str), g.speed_tm_vr_lag_s, color=ANIMAL_COL.get(a, "gray"), label=a)
axs[0].axhline(0.5, color="k", ls=":", lw=1); axs[0].set_ylabel("R²"); axs[0].set_title("TM vs VR speed fit quality")
axs[1].set_ylabel("lag (s)"); axs[1].set_title("TM vs VR clock offset")
for ax in axs:
    ax.tick_params(axis="x", rotation=45); ax.legend(fontsize=7)
plt.tight_layout(); save(fig, "3b_tm_vr_alignment_qc.png"); plt.show()

### Does a DCZ locomotor effect emerge over the session?

Split each DCZ/Saline session's rolling trajectory into the first vs second
half of its windows (same trial-window data used in §2) and compare —
mirrors the licking timecourse in §3 but for `run_speed_vr_cms` /
`decel_onset_cm`, which don't depend on licking at all.

In [ ]:
rows = []
for (a, s), g in T[T.session_label.isin(CONDITIONS)].groupby(["animal", "session_label"], observed=True):
    g = g.sort_values("trial_mid")
    if len(g) < 2:
        continue
    mid = g.trial_mid.median()
    early, late = g[g.trial_mid <= mid], g[g.trial_mid > mid]
    for k in ["run_speed_vr_cms", "decel_onset_cm", "anticipatory_index"]:
        rows.append(dict(animal=a, session_label=s, metric=k,
                         early=early[k].mean(), late=late[k].mean()))
ET = pd.DataFrame(rows)

fig, axs = plt.subplots(1, 3, figsize=(13, 4), squeeze=False)
for ax, k in zip(axs[0], ["run_speed_vr_cms", "decel_onset_cm", "anticipatory_index"]):
    sub = ET[ET.metric == k]
    for _, r in sub.iterrows():
        col = {"Saline": "tab:green", "DCZ100": "tab:orange", "DCZ200": "tab:red"}[r.session_label]
        ax.plot([0, 1], [r.early, r.late], "-o", ms=5, color=col,
                label=f"{r.animal} {r.session_label}", alpha=0.85)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["early half", "late half"])
    ax.set_title(k, fontsize=9)
axs[0, 0].legend(fontsize=6, ncol=2)
fig.suptitle("Early vs late half of session — does the effect build in?", y=1.02)
plt.tight_layout(); save(fig, "3b_early_vs_late_dcz.png"); plt.show()
ET.pivot_table(index=["animal", "session_label"], columns="metric", values=["early", "late"])

## 3c. Reviewer-requested analyses (2026-09-22)

Spatial-vs-temporal anticipation strategy (#3), dark-epoch vs. VR-approach
DCZ specificity (#4), the screen-transition trial classification (#5),
post-VR-onset drinking persistence (#6), black-screen running speed (#8),
and the design caveats + trial-level bootstrap (#9). #1/#2/#7 were already
covered by the existing sections above — see §7 at the end for the full
assessment against the original 9-point plan.

**Design caveat, stated here and repeated below because it matters every
time a DCZ-vs-Saline number appears anywhere in this notebook**: Saline →
DCZ100 → DCZ200 ran in fixed calendar order (06/26, 06/28, 06/30) — dose and
time-since-cohort-start are fully confounded in this dataset. A monotonic
DCZ100 < DCZ200 pattern is equally consistent with "bigger dose, bigger
effect" and "later in the sequence, bigger effect."

In [ ]:
def load_h5_group(rec, group):
    p = glob.glob(os.path.join(DATASET_ROOT, "*", rec, f"{rec}_lickmetrics.h5"))
    if not p:
        return None
    with h5py.File(p[0], "r") as f:
        if group not in f:
            return None
        g = f[group]
        out = {}
        for k in g:
            v = g[k][:]
            if v.dtype.kind in ("S", "O"):
                v = np.array([x.decode() if isinstance(x, bytes) else x for x in v])
            out[k] = v
        return pd.DataFrame(out)

### #3 — spatial vs. temporal anticipation strategy across sessions

R² of first-lick **position** vs. trial pace, and first-lick **time** vs.
trial pace (notebook 3 §5b), per session. The metric with the LOWER R² is
the more speed-invariant that session, i.e. the more likely controlling
variable.

In [ ]:
fig, axs = plt.subplots(1, len(ANIMALS), figsize=(7 * len(ANIMALS), 4.5), squeeze=False)
for ax, a in zip(axs[0], ANIMALS):
    g = M[M.animal == a].sort_values("session_label")
    x = [xpos[s] for s in g.session_label]
    ax.plot(x, g.spatial_temporal_r2_cm, "-o", ms=5, color="tab:orange", label="position ~ pace (R²)")
    ax.plot(x, g.spatial_temporal_r2_time, "-o", ms=5, color="tab:cyan", label="time ~ pace (R²)")
    ax.axvspan(xpos["Saline"] - 0.5, xpos["DCZ200"] + 0.5, color="tab:orange", alpha=0.06)
    ax.set_xticks(range(len(SESSIONS))); ax.set_xticklabels(SESSIONS, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("R²"); ax.set_title(f"{a} — spatial vs. temporal strategy"); ax.legend(fontsize=8)
plt.tight_layout(); save(fig, "3c_spatial_vs_temporal_strategy.png"); plt.show()
M[["animal", "session_label", "spatial_temporal_n", "spatial_temporal_r2_cm", "spatial_temporal_r2_time"]]

### #4 — is the DCZ effect specific to spatial (VR-approach) anticipation?

Compares the DCZ z-scores (vs. trained baseline) for a TEMPORAL/dark-epoch
metric (`psth_anticipatory_index`) against two SPATIAL/VR-approach metrics
(`lick_spread_cm`, `decel_onset_cm`). If the spatial metrics move
consistently while the temporal one doesn't, that's evidence the deficit is
specific to spatial prediction rather than a general motivation/arousal
change.

In [ ]:
spec_metrics = {"psth_anticipatory_index": "temporal\n(dark epoch)",
                "lick_spread_cm": "spatial\n(VR approach)",
                "decel_onset_cm": "spatial\n(VR approach, motor)"}
fig, axs = plt.subplots(1, len(ANIMALS), figsize=(6 * len(ANIMALS), 4.5), squeeze=False)
for ax, a in zip(axs[0], ANIMALS):
    x = np.arange(len(CONDITIONS)); w = 0.25
    for i, (k, lbl) in enumerate(spec_metrics.items()):
        ct = cond_table(k)
        r = ct[ct.animal == a]
        if not len(r):
            continue
        r = r.iloc[0]
        zs = [r[f"{s}_vs_trained_z"] for s in CONDITIONS]
        ax.bar(x + (i - 1) * w, zs, w, label=lbl)
    ax.axhline(0, color="k", lw=0.6); ax.axhspan(-1, 1, color="gray", alpha=0.1)
    ax.set_xticks(x); ax.set_xticklabels(CONDITIONS)
    ax.set_ylabel("z vs. trained baseline"); ax.set_title(f"{a} — specificity check")
axs[0, 0].legend(fontsize=7)
fig.suptitle("Dark (temporal) vs. VR-approach (spatial) DCZ specificity", y=1.02)
plt.tight_layout(); save(fig, "3c_dark_vs_spatial_specificity.png"); plt.show()

### #5 — spatial vs. transition-triggered trial proportions across sessions

Fraction of trials classified `spatial` (first anticipatory lick while still
running) vs. `transition_triggered` (first anticipatory lick only after the
black screen appeared) vs. `none`, per session.

In [ ]:
strat_cols = ["frac_trials_spatial_strategy", "frac_trials_transition_triggered", "frac_trials_no_anticipation"]
strat_lbl = ["spatial", "transition-triggered", "none"]
strat_col = ["tab:orange", "tab:cyan", "tab:gray"]
fig, axs = plt.subplots(1, len(ANIMALS), figsize=(7 * len(ANIMALS), 4.5), squeeze=False)
for ax, a in zip(axs[0], ANIMALS):
    g = M[M.animal == a].sort_values("session_label")
    x = np.arange(len(g)); bot = np.zeros(len(g))
    for c, lbl, col in zip(strat_cols, strat_lbl, strat_col):
        ax.bar(x, g[c], 0.8, bottom=bot, color=col, label=lbl)
        bot += g[c].to_numpy()
    ax.set_xticks(x); ax.set_xticklabels(g.session_label, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("fraction of trials"); ax.set_title(f"{a} — anticipation strategy")
axs[0, 0].legend(fontsize=8)
plt.tight_layout(); save(fig, "3c_strategy_proportions.png"); plt.show()

### #6 — does drinking persist past when VR should reappear?

Left: `frac_trials_post_vr_onset_licking` (fraction of trials with a lick
more than 1.5 s after the valve) — the direct per-trial version of the
already-suggestive `drink_bout_duration_s_median` finding (right; session-
level only, no within-session SD available).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2), squeeze=False)
xlab = ["trained\nB7-8"] + CONDITIONS

ct = cond_table("frac_trials_post_vr_onset_licking")
for _, r in ct.iterrows():
    y = [r.trained] + [r[s] for s in CONDITIONS]
    e = [r.baseline_sd] + [r[f"{s}_within_sd"] for s in CONDITIONS]
    axs[0, 0].errorbar(range(4), y, yerr=e, marker="o", ms=6, capsize=3,
                       color=ANIMAL_COL.get(r.animal, "gray"), label=r.animal)
axs[0, 0].set_xticks(range(4)); axs[0, 0].set_xticklabels(xlab, fontsize=8)
axs[0, 0].set_title("frac_trials_post_vr_onset_licking", fontsize=9); axs[0, 0].legend(fontsize=8)

for a in ANIMALS:
    gm = M[M.animal == a].set_index("session_label")
    base = gm.loc[gm.index.isin(BASELINE), "drink_bout_duration_s_median"]
    trained = gm.loc[gm.index.isin(TRAINED), "drink_bout_duration_s_median"].mean()
    y = [trained] + [gm.loc[s, "drink_bout_duration_s_median"] if s in gm.index else np.nan for s in CONDITIONS]
    axs[0, 1].plot(range(4), y, "-o", ms=6, color=ANIMAL_COL.get(a, "gray"), label=a)
    axs[0, 1].axhspan(trained - base.std(), trained + base.std(), color=ANIMAL_COL.get(a, "gray"), alpha=0.08)
axs[0, 1].set_xticks(range(4)); axs[0, 1].set_xticklabels(xlab, fontsize=8)
axs[0, 1].set_title("drink_bout_duration_s_median", fontsize=9)

fig.suptitle("Post-reward consummatory persistence", y=1.02)
plt.tight_layout(); save(fig, "3c_post_vr_onset_persistence.png"); plt.show()

### #8 — running speed during the black screen (TM log)

The TM log is a separate hardware stream sampled continuously regardless of
VR state, so it can see movement during the pre-valve dark period that the
VR position log structurally can't. **Caution**: the TM→cm/s calibration is
fit on VR-visible transit data; applying it to the black-screen epoch
extrapolates outside that fit's range, so the absolute cm/s number is less
trustworthy than the cross-condition comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for a in ANIMALS:
    gm = M[M.animal == a].set_index("session_label")
    base = gm.loc[gm.index.isin(BASELINE), "black_screen_speed_cms"]
    trained = gm.loc[gm.index.isin(TRAINED), "black_screen_speed_cms"].mean()
    y = [trained] + [gm.loc[s, "black_screen_speed_cms"] if s in gm.index else np.nan for s in CONDITIONS]
    ax.plot(range(4), y, "-o", ms=6, color=ANIMAL_COL.get(a, "gray"), label=a)
    ax.axhspan(trained - base.std(), trained + base.std(), color=ANIMAL_COL.get(a, "gray"), alpha=0.08)
ax.set_xticks(range(4)); ax.set_xticklabels(["trained\nB7-8"] + CONDITIONS, fontsize=9)
ax.set_ylabel("cm/s"); ax.set_title("Black-screen (pre-valve) running speed"); ax.legend(fontsize=9)
plt.tight_layout(); save(fig, "3c_black_screen_speed.png"); plt.show()

### #9a — baseline plateau check (is B6–B8 stable?)

Crude check: the B6/B7/B8 spread relative to the across-baseline (B1–B8) SD
(< ~1 suggests a plateau; > ~1 suggests the animal is still changing at the
end of baseline) — and how far Saline sits from that plateau, same units.

In [ ]:
plateau_days = ["B6", "B7", "B8"]
rows = []
for a in ANIMALS:
    gm = M[M.animal == a].set_index("session_label")
    base_sd = gm.loc[gm.index.isin(BASELINE), list(KEY_METRICS)].std()
    plateau_vals = gm.loc[gm.index.isin(plateau_days), list(KEY_METRICS)]
    plateau_range = plateau_vals.max() - plateau_vals.min()
    has_saline = "Saline" in gm.index
    for k in KEY_METRICS:
        sdev = ((gm.loc["Saline", k] - plateau_vals[k].mean()) / base_sd[k]
               if has_saline and base_sd[k] else np.nan)
        rows.append(dict(animal=a, metric=k,
                         plateau_range_over_baseline_sd=float(plateau_range[k] / base_sd[k]) if base_sd[k] else np.nan,
                         saline_dev_over_baseline_sd=float(sdev) if np.isfinite(sdev) else np.nan))
PLAT = pd.DataFrame(rows)
print("plateau_range_over_baseline_sd: B6-B8 spread / across-baseline SD (<~1 ~ plateau)")
print("saline_dev_over_baseline_sd: Saline's distance from the B6-8 plateau mean, same units")
PLAT.pivot_table(index="metric", columns="animal",
                 values=["plateau_range_over_baseline_sd", "saline_dev_over_baseline_sd"]).round(2)

### #9b — trial-level bootstrap against pooled late-baseline (B7+B8) trials

Upgrades the `(condition − trained) / baseline_SD` z-score heuristic used
throughout this notebook with an actual resampling test: pool every trial
from B7+B8 per animal as the late-baseline reference; for each condition,
draw (with replacement) as many trials as that session actually has, 2000
times, and see where the condition's own observed value falls in that null
distribution (empirical two-tailed p-value). Scoped to metrics that are
cleanly trial-resamplable without re-deriving a position-binned curve —
`decel_onset_cm` is a position-binned aggregate across many trials at once,
not a simple per-trial statistic, so it isn't bootstrapped here.

In [ ]:
def per_trial_arrays(licks_full, trials_full, cm_per_au, rz_cm):
    ap = licks_full[licks_full.category.isin(["approaching_nonreward", "approaching_reward"])]
    by_trial = {t: g.position_au.to_numpy() * cm_per_au for t, g in ap.groupby("trial")}
    cm_list, inrz_list, speed_list = [], [], []
    for t in trials_full.trial:
        pos = by_trial.get(t, np.array([]))
        cm_list.append(pos)
        inrz_list.append((pos >= rz_cm - 30).astype(float))
        spd = trials_full.loc[trials_full.trial == t, "approach_speed_cms"].to_numpy()
        speed_list.append(spd[np.isfinite(spd)])
    return cm_list, inrz_list, speed_list

BOOT_SPEC = {"lick_spread_cm": np.std, "frac_approach_licks_in_rz": np.mean, "run_speed_vr_cms": np.mean}
N_BOOT = 2000
rng = np.random.default_rng(0)
boot_rows = []
for a in ANIMALS:
    gm = M[M.animal == a].set_index("session_label")
    if not all(s in gm.index for s in TRAINED):
        continue
    cm_per_au = float(gm.loc[TRAINED[0], "cm_per_au"])
    rz_cm = float(gm.loc[TRAINED[0], "reward_zone_cm"])

    ref_cm, ref_inrz, ref_speed = [], [], []
    for s in TRAINED:
        lf, tf = load_h5_group(gm.loc[s, "recording"], "licks_full"), load_h5_group(gm.loc[s, "recording"], "trials_full")
        if lf is None or tf is None:
            continue
        c, r, sp = per_trial_arrays(lf, tf, cm_per_au, rz_cm)
        ref_cm += c; ref_inrz += r; ref_speed += sp

    for s in CONDITIONS:
        if s not in gm.index:
            continue
        lf, tf = load_h5_group(gm.loc[s, "recording"], "licks_full"), load_h5_group(gm.loc[s, "recording"], "trials_full")
        if lf is None or tf is None:
            continue
        cond_cm, cond_inrz, cond_speed = per_trial_arrays(lf, tf, cm_per_au, rz_cm)
        n_draw = len(tf)

        for k, (ref, cond) in zip(BOOT_SPEC, [(ref_cm, cond_cm), (ref_inrz, cond_inrz), (ref_speed, cond_speed)]):
            stat = BOOT_SPEC[k]
            null = trial_cluster_bootstrap(ref, n_draw, lambda x: stat(x), n_boot=N_BOOT, rng=rng)
            pooled = np.concatenate([x for x in cond if len(x)]) if any(len(x) for x in cond) else np.array([])
            observed = float(stat(pooled)) if len(pooled) else np.nan
            p = bootstrap_pvalue(observed, null)
            boot_rows.append(dict(animal=a, session_label=s, metric=k, observed=observed,
                                  null_median=float(np.nanmedian(null)),
                                  null_lo=float(np.nanpercentile(null[np.isfinite(null)], 2.5)) if np.isfinite(null).any() else np.nan,
                                  null_hi=float(np.nanpercentile(null[np.isfinite(null)], 97.5)) if np.isfinite(null).any() else np.nan,
                                  p_value=p))
BOOT = pd.DataFrame(boot_rows)
out = os.path.join(DATASET_ROOT, "trial_bootstrap_summary.csv")
BOOT.to_csv(out, index=False)
print("wrote", out)
BOOT.round(3)

In [ ]:
fig, axs = plt.subplots(len(ANIMALS), len(BOOT_SPEC), figsize=(4.5 * len(BOOT_SPEC), 4 * len(ANIMALS)), squeeze=False)
for i, a in enumerate(ANIMALS):
    for j, k in enumerate(BOOT_SPEC):
        ax = axs[i, j]
        sub = BOOT[(BOOT.animal == a) & (BOOT.metric == k)]
        for _, r in sub.iterrows():
            col = {"Saline": "tab:green", "DCZ100": "tab:orange", "DCZ200": "tab:red"}[r.session_label]
            # error bar = the NULL distribution's 95% CI (centred on its own
            # median, so the two half-widths are always >= 0 by construction);
            # `observed` is a separate point that can fall anywhere relative
            # to it, including outside the bar -- that's the whole point of
            # the test, so it must not be forced inside an errorbar around
            # itself (that produces a negative half-width whenever the
            # effect is large enough to matter).
            ax.errorbar([r.session_label], [r.null_median],
                       yerr=[[r.null_median - r.null_lo], [r.null_hi - r.null_median]],
                       fmt="none", ecolor="gray", capsize=4, alpha=0.5)
            ax.scatter([r.session_label], [r.observed], color=col, s=60, zorder=3,
                      label=f"p={r.p_value:.3f}")
            ax.scatter([r.session_label], [r.null_median], color="k", marker="_", s=200, zorder=2)
        ax.set_title(f"{a} — {k}", fontsize=9)
        for _, r in sub.iterrows():
            ax.annotate(f"p={r.p_value:.3f}", (r.session_label, r.observed), fontsize=7,
                       xytext=(3, 3), textcoords="offset points")
fig.suptitle("Trial-level bootstrap: observed value vs. late-baseline (B7+B8) null "
            "(gray bar = null 95% CI, dotted line = null median)", y=1.01)
plt.tight_layout(); save(fig, "3c_trial_bootstrap.png"); plt.show()

## 4. Effect-size summary table

In [ ]:
summ = []
for k in {**KEY_METRICS, **CONTROL_METRICS}:
    ct = cond_table(k)
    for _, r in ct.iterrows():
        summ.append(dict(metric=k, animal=r.animal,
                         baseline_mean=r.baseline_mean, baseline_sd=r.baseline_sd,
                         trained=r.trained,
                         saline=r.Saline, dcz100=r.DCZ100, dcz200=r.DCZ200,
                         saline_z=r["Saline_vs_trained_z"],
                         dcz100_z=r["DCZ100_vs_trained_z"],
                         dcz200_z=r["DCZ200_vs_trained_z"]))
summary = pd.DataFrame(summ)
out = os.path.join(DATASET_ROOT, "session_comparison_summary.csv")
summary.to_csv(out, index=False)
print("wrote", out)
print("figures ->", FIGDIR)
summary.round(3)

## 5. Learning-rate curve fit across B1–B8, and the position-freeze confound

Rather than just early-vs-late, fit an exponential approach
`y = y0 + (y_inf - y0)*(1 - exp(-session/tau))` to `anticipatory_index` and
`lick_spread_cm` over the 8 baseline days per animal, to get a single
learning-rate number (`tau`, in sessions) instead of a description.

Separately: `pct_position_frozen` (the VR-log-freeze artifact, [[project_licking_behavior]])
also climbs B1→DCZ200 for both animals, independent of drug — check whether
it's *correlated* with the learning metrics, since a confound that trends
the same direction as learning could inflate or mask the DCZ comparisons above.

In [ ]:
from scipy.optimize import curve_fit

def exp_approach(x, y0, yinf, tau):
    return y0 + (yinf - y0) * (1 - np.exp(-x / max(tau, 1e-3)))

fig, axs = plt.subplots(1, len(ANIMALS), figsize=(6.5 * len(ANIMALS), 4.2), squeeze=False)
fit_rows = []
for ax, a in zip(axs[0], ANIMALS):
    g = M[(M.animal == a) & (M.session_label.isin(BASELINE))].sort_values("session_label")
    x = np.arange(1, len(g) + 1, dtype=float)
    y = g["anticipatory_index"].to_numpy()
    ax.plot(x, y, "o", color=ANIMAL_COL.get(a, "gray"))
    ok = np.isfinite(y)
    tau = np.nan
    if ok.sum() >= 4:
        try:
            p0 = [y[ok][0], y[ok][-1], 3.0]
            popt, _ = curve_fit(exp_approach, x[ok], y[ok], p0=p0, maxfev=5000)
            xx = np.linspace(1, len(g), 50)
            ax.plot(xx, exp_approach(xx, *popt), "-", color=ANIMAL_COL.get(a, "gray"), lw=1.5)
            tau = float(popt[2])
        except RuntimeError:
            pass
    ax.set_xlabel("baseline day (B1..B8)"); ax.set_ylabel("anticipatory_index")
    ax.set_title(f"{a} — learning-rate fit  (tau = {tau:.1f} sessions)" if np.isfinite(tau)
                 else f"{a} — fit did not converge")
    fit_rows.append(dict(animal=a, metric="anticipatory_index", tau_sessions=tau))
plt.tight_layout(); save(fig, "5_learning_rate_fit.png"); plt.show()
pd.DataFrame(fit_rows)

In [ ]:
# freeze-fraction confound: is pct_position_frozen correlated with the metrics
# that are supposed to reflect learning / DCZ effects?
conf_metrics = ["anticipatory_index", "lick_spread_cm", "run_speed_vr_cms"]
rows = []
for k in conf_metrics:
    d = M.dropna(subset=[k, "pct_position_frozen"])
    r = np.corrcoef(d.pct_position_frozen, d[k])[0, 1] if len(d) > 3 else np.nan
    rows.append(dict(metric=k, r_vs_pct_frozen=r, n=len(d)))
CONF = pd.DataFrame(rows)
print("correlation with pct_position_frozen (all 22 sessions, both animals pooled):")
print(CONF.to_string(index=False))

fig, axs = plt.subplots(1, len(conf_metrics), figsize=(5 * len(conf_metrics), 4), squeeze=False)
for ax, k in zip(axs[0], conf_metrics):
    for a in ANIMALS:
        g = M[M.animal == a]
        ax.scatter(g.pct_position_frozen, g[k], color=ANIMAL_COL.get(a, "gray"), label=a, s=25)
    ax.set_xlabel("% session frozen"); ax.set_ylabel(k)
axs[0, 0].legend(fontsize=8)
fig.suptitle("Freeze-fraction confound check", y=1.02)
plt.tight_layout(); save(fig, "5_freeze_confound.png"); plt.show()

---
### Reading the summary

- **`*_z`** = (condition − trained-baseline) in units of the across-baseline-day SD.
  |z| ≲ 1 → within normal day-to-day variation; large negative z on the narrowing
  metrics (or positive on `lick_spread_cm`) → DCZ broadened the anticipatory licking.
- **Saline** should sit near 0 (sanity check on the manipulation).
- With **n = 1 session / condition / animal**, treat a consistent direction *across
  both animals* and *dose-ordered* (DCZ200 > DCZ100) as the evidence, not significance.
- Trials flagged in `led_flagged_trials` used the VR-log event times (LED disagreed);
  `led_blocked_runs` marks stretches where the LED was occluded — check the QC table.

## 6. Presentation summary

Consolidated, larger-format versions of the headline result for external use
(meetings, slides) — pulls together the single clearest visual (the lick-rate
heatmap going dark under DCZ) with the three metrics that are consistent
*across both animals* and *dose-ordered*: `lick_spread_cm`, `decel_onset_cm`,
`frac_approach_licks_in_rz`. Simple rate metrics (`anticipatory_index`,
`rz_lick_rate_hz`) are deliberately left out here — they move inconsistently
across animals/conditions and would undercut the headline if included without
the caveats in §4/§5 above. **n = 1 session/condition/animal — say that up
front, not as a fallback answer.**

In [ ]:
HEADLINE_METRICS = {
    "lick_spread_cm":              "spatial spread of anticipatory licks (cm)  —  higher = less precise",
    "decel_onset_cm":               "deceleration-onset position (cm)  —  lower = earlier/further from reward",
    "frac_approach_licks_in_rz":    "fraction of approach licks landing in the reward zone",
}

# --- standalone, larger lick-rate heatmap (the single most visual figure) ---
fig, axs = plt.subplots(1, len(ANIMALS), figsize=(8 * len(ANIMALS), 6), squeeze=False)
for ax, a in zip(axs[0], ANIMALS):
    g = M[M.animal == a].sort_values("session_label")
    rz = g.reward_zone_cm.median()
    s1, b1, H1 = session_matrix(g, curves_cache[a], lambda c: c.rate_hz.to_numpy())
    heatmap_panel(ax, s1, b1, H1, rz, f"{a}", "approach lick rate (Hz)", fig=fig)
    ax.tick_params(labelsize=10)
    ax.title.set_fontsize(13)
fig.suptitle("Anticipatory lick rate fades under DCZ  (B1 top -> DCZ200 bottom, each animal)",
            fontsize=14, y=1.02)
plt.tight_layout(); save(fig, "6_headline_heatmap.png"); plt.show()

In [ ]:
# --- headline metrics: trained baseline vs Saline / DCZ100 / DCZ200 ---
fig, axs = plt.subplots(1, len(HEADLINE_METRICS), figsize=(6 * len(HEADLINE_METRICS), 5), squeeze=False)
xlab = ["trained\n(B7-8)"] + CONDITIONS
for ax, k in zip(axs[0], HEADLINE_METRICS):
    ct = cond_table(k)
    for _, r in ct.iterrows():
        y = [r.trained] + [r[s] for s in CONDITIONS]
        e = [r.baseline_sd] + [r[f"{s}_within_sd"] for s in CONDITIONS]
        ax.errorbar(range(4), y, yerr=e, marker="o", ms=8, lw=2, capsize=4,
                    color=ANIMAL_COL.get(r.animal, "gray"), label=r.animal)
    ax.set_xticks(range(4)); ax.set_xticklabels(xlab, fontsize=10)
    ax.set_title(k, fontsize=13)
    ax.set_xlabel(HEADLINE_METRICS[k], fontsize=8.5, wrap=True)
axs[0, 0].legend(fontsize=10, loc="best")
fig.suptitle("Precision of anticipatory approach — consistent across both animals, dose-ordered",
            fontsize=14, y=1.03)
plt.tight_layout(); save(fig, "6_headline_metrics.png"); plt.show()

In [ ]:
# --- one combined slide: heatmap on top, headline metrics on bottom ---
fig = plt.figure(figsize=(16, 11))
gs = fig.add_gridspec(2, max(len(ANIMALS), len(HEADLINE_METRICS)),
                      height_ratios=[1.1, 1], hspace=0.45, wspace=0.5)

for j, a in enumerate(ANIMALS):
    ax = fig.add_subplot(gs[0, j])
    g = M[M.animal == a].sort_values("session_label")
    rz = g.reward_zone_cm.median()
    s1, b1, H1 = session_matrix(g, curves_cache[a], lambda c: c.rate_hz.to_numpy())
    heatmap_panel(ax, s1, b1, H1, rz, f"{a} — approach lick rate", "Hz", fig=fig)

for j, k in enumerate(HEADLINE_METRICS):
    ax = fig.add_subplot(gs[1, j])
    ct = cond_table(k)
    for _, r in ct.iterrows():
        y = [r.trained] + [r[s] for s in CONDITIONS]
        e = [r.baseline_sd] + [r[f"{s}_within_sd"] for s in CONDITIONS]
        ax.errorbar(range(4), y, yerr=e, marker="o", ms=7, lw=1.8, capsize=3,
                    color=ANIMAL_COL.get(r.animal, "gray"), label=r.animal)
    ax.set_xticks(range(4)); ax.set_xticklabels(xlab, fontsize=8)
    ax.set_title(k, fontsize=10.5)
    if j == 0:
        ax.legend(fontsize=8)

fig.suptitle("DCZ disrupts the spatial precision of anticipatory licking\n"
            "(n=1 session/condition/animal — descriptive, consistent direction across animals)",
            fontsize=15, y=1.0)
save(fig, "6_summary_slide.png"); plt.show()

## 7. Proposed next analyses (reviewer notes, 2026-09-22) — status

Trial structure assumed below: VR approach (position-indexed) → ~1.5 s
pre-valve black screen → valve → ~1.5 s post-valve black screen (bout can run
past this into the next trial's VR onset), water delivered every trial
regardless of licking (so pre-valve licking is anticipatory, not operant).
**All 9 items built as of 2026-09-22** — §3c above holds the new
cross-session figures/tables; notebook 3 §5b/§5c/§4/§4b hold the new
per-recording ones. Status + what each shows, per item:

### 1. Epoch bookkeeping — done
Consummatory scatter panels (notebook 3 §3) now mark the ±1.5 s epoch
boundaries explicitly instead of silently windowing to ±2.5 s. Confirmed
there's no actual double-counting (`classify_licks` assigns each lick to
exactly one category). Added a live QC check for the ~5 cm carry-over
cluster in the approach panel (the `carryover_frames=45` fix, 2026-09-15,
should already have removed it — the notebook now prints whether it's
actually gone on each run instead of assuming so).

### 2. Spatial anticipation during VR — already covered
`lick_rate_vs_position`, `frac_approach_licks_in_rz`, `first_approach_lick`.

### 3. Spatial vs. temporal anticipation — done, §3c above + notebook 3 §5b
`spatial_temporal_onset_table` / `spatial_vs_temporal_fit` regress first-lick
position and first-lick time each against trial pace (natural speed
variation); whichever has the lower R² is the more speed-invariant, hence
more likely controlling variable, per session.

### 4. Anticipation in the dark — done, §3c #4 above + notebook 3 §4
`frac_trials_with_prevalve_lick` added. The dark-vs-spatial DCZ-specificity
comparison (§3c #4 above) makes the suggestive pattern noted here explicit:
`psth_anticipatory_index` (dark/temporal) does NOT move cleanly with DCZ
while `lick_spread_cm`/`decel_onset_cm` (spatial) DO, cross-animal and
dose-ordered — see that figure for the actual numbers, not just the claim.

### 5. Screen transition as a lick trigger — done, §3c #5 above + notebook 3 §5c
`classify_trial_anticipation_strategy` tags every trial spatial /
transition_triggered / none; §3c #5 tracks the proportions across sessions.

### 6. Consummatory bout structure — done, §3c #6 above + notebook 3 §4b
`frac_trials_post_vr_onset_licking` is the direct per-trial version of the
`drink_bout_duration_s_median` finding (DCZ increases bout duration vs.
trained baseline in both animals — JSY083 trained ≈1.3 s → DCZ100 2.75 s,
DCZ200 2.0 s; JSY084 trained ≈1.6 s → DCZ100 2.17 s, DCZ200 1.97 s); ILI
distribution added to notebook 3 §4b.

### 7. Within-session engagement, per epoch — already covered
`rolling_metrics` + the approach-vs-consummatory baseline figure (§1 above)
already separate approach from consummatory_pre_reward across trial windows;
`consummatory_post_reward_per_trial` and `run_speed_vr_cms` added to notebook
3 §8's rolling-trajectory panel for the full 3-epoch + locomotor picture.

### 8. Speed covariates — done, §3c #8 above + notebook 3 §2b
`black_screen_running_speed` uses the TM log (a separate hardware stream
sampled continuously regardless of VR state) during the pre-valve dark
epoch. Consistently ~20 cm/s across every session checked (both animals, all
conditions) — notably HIGHER than the VR-visible transit speed (~7-12 cm/s),
which is either a real "faster without visual pacing" effect or an artifact
of extrapolating the TM→cm/s calibration outside the range it was fit on
(fit on VR-visible transit data only) — flagged as a caution in §3c #8, not
resolved.

### 9. Baseline plateau and design caveats — done, §3c #9a/#9b above
- **Saline → DCZ100 → DCZ200 ran in fixed calendar order (06/26, 06/28,
  06/30) — dose and time-since-start are fully confounded in this dataset.**
  Now stated explicitly at the top of §3c, not just here.
- #9a: a crude B6-B8 plateau check (spread relative to across-baseline SD)
  + how far Saline sits from that plateau, per metric per animal.
- #9b: trial-level bootstrap against pooled B7+B8 trials, empirical
  two-tailed p-values, scoped to `lick_spread_cm` / `frac_approach_licks_in_rz`
  / `run_speed_vr_cms` (metrics that are cleanly trial-resamplable without
  re-deriving a position-binned curve — `decel_onset_cm` is a
  many-trials-at-once aggregate and isn't bootstrapped here). Also written to
  `trial_bootstrap_summary.csv`.
- A washout session (design note for future cohorts, nothing to implement on
  current data) would still be the only way to actually separate dose from
  order.

### Imaging implication
The −1.5…0 s dark epoch is a window with no visual drive but (if water is
delivered every trial) real reward expectation — if RSC input carries
expectation signals into V1, DCZ's effect on V1 activity specifically in this
epoch is a cleaner test than anything during the visually-driven VR approach,
since it avoids spatial-visual-feature confounds. Not an analysis on the
current behavior-only dataset, but the reason #3/#4 above matter beyond
licking behavior itself.